In [ ]:
# 0) Importing packages/libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

from IPython.display import display

import graphviz

In [ ]:
# 1) Loading data in

df = pd.read_csv("CustomerChurn.csv")

# Quick EDA
display(df.head())
display(df.tail())
print(df.dtypes)
display(df.describe(include="all"))

In [ ]:
# 2) EDA

# Correlation heatmap (numeric only)
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="OrRd", linewidths=0.5)
plt.title("Feature Correlation Heatmap (Numeric)")
plt.show()

# Status percentage bar chart
status_counts = df["Status"].value_counts(normalize=True) * 100
plt.figure(figsize=(6, 4))
ax = sns.barplot(x=status_counts.index, y=status_counts.values, palette="pastel")

for i, p in enumerate(status_counts.values):
    ax.text(i, p + 1, f"{p:.1f}%", ha="center", fontsize=12, fontweight="bold")

plt.title("Status Percentage", fontsize=14, fontweight="bold")
plt.xlabel("Status", fontsize=12)
plt.ylabel("Percentage", fontsize=12)
plt.xticks(ticks=[0, 1], labels=["Active (1)", "Inactive (2)"], fontsize=12)
plt.ylim(0, 100)
plt.show()

# Correlation with churn
churn_corr = numeric_df.drop(columns=["Churn"], errors="ignore").corrwith(numeric_df["Churn"])
churn_corr.sort_values().plot(kind="bar", grid=True, figsize=(10, 6), title="Variable Correlation w/ Churn")
plt.show()


In [ ]:
# 3) Cleaning

# Dropping columns that were not useful / too correlated
DROP_COLS = ["Customer Value", "Age Group", "Complains", "Seconds of Use"]

df_model = df.drop(columns=DROP_COLS, errors="ignore").copy()

# Split features/target
X = df_model.drop(columns=["Churn"])
y = df_model["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

In [ ]:
# 4) Decision Tree model

clf = DecisionTreeClassifier(max_depth=12, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="g", xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Decision tree visualization (inline)
dot_data = export_graphviz(
    clf,
    out_file=None,
    filled=True,
    feature_names=X.columns,
    class_names=["No Churn", "Churn"],
    rounded=True,
    special_characters=True,
)
graph = graphviz.Source(dot_data)
graph  # shows in notebook

#outputting as pdf
graph.render("decision_tree", format="pdf", cleanup=True)

In [ ]:
# 5) Feature importance

importances = clf.feature_importances_
feat_importance = (
    pd.DataFrame({"Feature": X.columns, "Importance": importances})
      .sort_values(by="Importance", ascending=False)
)

plt.figure(figsize=(10, 6))
ax = sns.barplot(x="Importance", y="Feature", data=feat_importance, palette="Blues_r")

for container in ax.containers:
    ax.bar_label(container, fmt="%.4f", padding=5)

plt.title("Feature Importance from Decision Tree")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

display(feat_importance)

In [ ]:
# 6) K-Means clustering using top features

TOP_N = 7
top_features = feat_importance["Feature"].head(TOP_N).tolist()

X_cluster = df_model[top_features].copy()

scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)

clusters = kmeans.fit_predict(X_cluster_scaled) + 1

# Attaching cluster labels
df_clustered = df_model.copy()
df_clustered["Churn_Cluster"] = clusters

print("Cluster Counts:\n", df_clustered["Churn_Cluster"].value_counts().sort_index())

# Churn rate per cluster
churn_rates = df_clustered.groupby("Churn_Cluster")["Churn"].mean() * 100

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=churn_rates.index, y=churn_rates.values, palette="coolwarm")

for i, rate in enumerate(churn_rates.values):
    ax.text(i, rate + 1, f"{rate:.1f}%", ha="center", fontsize=12, fontweight="bold")

plt.xlabel("Cluster")
plt.ylabel("Churn Rate (%)")
plt.title("Churn Rate per Cluster")
plt.ylim(0, max(churn_rates.values) + 5)
plt.show()

# Summary stats (means of top features)
cluster_summary = df_clustered.groupby("Churn_Cluster")[top_features].mean()
display(cluster_summary.style.format("{:.2f}"))

print("\nChurn Rate per Cluster (%):")
print(churn_rates.to_string(float_format="%.2f"))


In [ ]:
# 7) Mean/Median summary for selected features (optional)
features_to_check = [
    "Status",
    "Frequency of use",
    "Subscription  Length",
    "Frequency of SMS",
    "Distinct Called Numbers",
    "Call  Failure",
    "Age",
]

mean_table = df_clustered.groupby("Churn_Cluster")[features_to_check].mean()
median_table = df_clustered.groupby("Churn_Cluster")[features_to_check].median()

print("Cluster Summary (Mean):")
display(mean_table)

print("\nCluster Summary (Median):")
display(median_table)